# Individual Statistics and Mapping

Analyze individual encounter patterns over the past year and visualize them on an interactive map:

1. Count distinct individuals grouped by species
2. Identify the most frequently encountered individuals
3. Map all encounters for the top 5 individuals with color-coding

**Extra dependency** — install before running this notebook:

```bash
uv pip install ipyleaflet
```

Set the usual environment variables (`WILDBOOK_URL`, `WILDBOOK_USERNAME`,
`WILDBOOK_PASSWORD`) before starting the kernel, or pass credentials
explicitly to `client.login()`.

In [1]:
# Import required libraries
import os
import getpass
from datetime import datetime, timedelta
from collections import Counter, defaultdict

from dotenv import load_dotenv
from pywildbook import WildbookClient
from pywildbook.queries import filter_by_date_range, filter_by_individual, combine_queries
from ipyleaflet import Map, CircleMarker, Popup
from ipywidgets import HTML, VBox, HBox, Layout

# Load environment variables from .env file if present
load_dotenv()

# Prompt for credentials if not set in environment
if not os.environ.get("WILDBOOK_URL"):
    os.environ["WILDBOOK_URL"] = input("WILDBOOK_URL: ")
if not os.environ.get("WILDBOOK_USERNAME"):
    os.environ["WILDBOOK_USERNAME"] = input("WILDBOOK_USERNAME: ")
if not os.environ.get("WILDBOOK_PASSWORD"):
    os.environ["WILDBOOK_PASSWORD"] = getpass.getpass("WILDBOOK_PASSWORD: ")

In [2]:
# Authenticate with Wildbook
client = WildbookClient()
user = client.login()
print(f"Logged in as {user['username']}")

Logged in as kirk


In [3]:
# Define helper functions for species names and color generation

def get_species_name(enc):
    """Extract species name from encounter, preferring taxonomy field.

    The taxonomy field is often more reliable than separate genus/specificEpithet fields.
    Falls back to combining genus and species if taxonomy is not available.
    """
    # Try taxonomy field first (most reliable)
    taxonomy = enc.get('taxonomy', '').strip()
    if taxonomy:
        return taxonomy

    # Fall back to genus + specificEpithet
    genus = enc.get('genus', '').strip()
    species = enc.get('specificEpithet', '').strip()
    if genus or species:
        return f"{genus} {species}".strip()

    return 'Unknown'


def generate_color(individual_id):
    """Generate a consistent color for an individual based on their ID.

    Uses hashing to ensure the same individual always gets the same color.
    Returns gray for unassigned encounters (no individual ID).
    """
    if not individual_id:
        return '#888888'  # Gray for unassigned encounters

    # Use hash to generate RGB values
    hash_val = hash(individual_id)
    r = (hash_val & 0xFF0000) >> 16
    g = (hash_val & 0x00FF00) >> 8
    b = hash_val & 0x0000FF

    # Ensure colors are vibrant (avoid too dark or too light)
    r = max(50, min(200, r))
    g = max(50, min(200, g))
    b = max(50, min(200, b))

    return f'#{r:02x}{g:02x}{b:02x}'

In [4]:
# Search for encounters from the past year to identify active individuals
one_year_ago = (datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d')
print(f"Searching for encounters since {one_year_ago}...")

results = client.search_encounters(
    filter_by_date_range(start_date=one_year_ago),
    size=1000,  # Adjust if you expect more encounters
    sort='date',
    sort_order='desc'
)

encounters = results.get('hits', [])
print(f"Found {len(encounters)} encounters from the past year")

Searching for encounters since 2025-04-23...
Found 897 encounters from the past year


In [5]:
# Analysis 1: Group individuals by species
# This counts unique individuals per species (not total encounters)

individuals_by_species = defaultdict(set)

for enc in encounters:
    individual_id = enc.get('individualId')
    if not individual_id:
        continue

    species_name = get_species_name(enc)
    individuals_by_species[species_name].add(individual_id)

print("\n=== Distinct Individuals by Species ===")
print(f"{'Species':<30} {'Count':>10}")
print("-" * 42)

for species_name in sorted(individuals_by_species.keys()):
    count = len(individuals_by_species[species_name])
    print(f"{species_name:<30} {count:>10}")

total_individuals = sum(len(ids) for ids in individuals_by_species.values())
print("-" * 42)
print(f"{'Total':<30} {total_individuals:>10}")


=== Distinct Individuals by Species ===
Species                             Count
------------------------------------------
Bombina variegata                      50
Giraffa giraffa                         1
Giraffa giraffa giraffa                 4
Salamandra salamandra                 675
------------------------------------------
Total                                 730


In [6]:
# Analysis 2: Find most frequently encountered individuals
# This counts encounters per individual to identify the most-sighted individuals

individual_encounters = Counter()
individual_details = {}  # Store species and display name for each individual

for enc in encounters:
    individual_id = enc.get('individualId')
    if not individual_id:
        continue

    individual_encounters[individual_id] += 1

    # Store details (first encounter wins)
    if individual_id not in individual_details:
        species_name = get_species_name(enc)
        display_name = enc.get('individualDisplayName', individual_id)
        individual_details[individual_id] = {
            'species': species_name,
            'display_name': display_name
        }

print("\n=== Most Frequently Encountered Individuals ===")
print(f"{'Display Name':<30} {'Species':<30} {'Encounters':>12}")
print("-" * 74)

# Store top 5 for mapping
top_5_individuals = []
for individual_id, count in individual_encounters.most_common(10):
    details = individual_details.get(individual_id, {'species': 'Unknown', 'display_name': individual_id})
    display_name = details['display_name']
    species_name = details['species']
    print(f"{display_name:<30} {species_name:<30} {count:>12}")

    # Save top 5 for mapping
    if len(top_5_individuals) < 5:
        top_5_individuals.append(individual_id)


=== Most Frequently Encountered Individuals ===
Display Name                   Species                          Encounters
--------------------------------------------------------------------------
A-A-C-0007a-a-a-a-a-a-a-a-a-a-a-a- Salamandra salamandra                     8
A-A-C-0005                     Salamandra salamandra                     5
sitaaa                         Salamandra salamandra                     5
A-A-C-0015                     Salamandra salamandra                     4
BGBI_22-2100                   Salamandra salamandra                     3
BUWK2769                       Salamandra salamandra                     3
erintest                       Giraffa giraffa giraffa                   2
Gerry                          Giraffa giraffa giraffa                   2
BUWK2738                       Salamandra salamandra                     2
LB-KoB-4-4                     Salamandra salamandra                     2


In [7]:
# Fetch ALL encounters for the top 5 individuals
# This gives us their complete encounter history for better visualization

print("\nFetching all encounters for top 5 individuals...")

# Build a query that combines filters for all 5 individuals using OR logic
individual_queries = [filter_by_individual(ind_id) for ind_id in top_5_individuals]
top_5_query = combine_queries(*individual_queries, operator='should')

# Search for all encounters of these individuals (no date filter)
all_encounters_top_5 = client.search_encounters(
    top_5_query,
    size=500,  # Adjust if individuals have many encounters
    sort='date',
    sort_order='desc'
)

map_encounters = all_encounters_top_5.get('hits', [])
print(f"Found {len(map_encounters)} total encounters for these {len(top_5_individuals)} individuals")


Fetching all encounters for top 5 individuals...
Found 33 total encounters for these 5 individuals


In [8]:
# Define popup content for map markers
# Each marker will show detailed encounter information when clicked

def create_popup(enc):
    """Create a popup with encounter details for display on the map.

    Includes species, individual name, year, location, and encounter ID.
    """
    genus = enc.get('genus', '')
    species = enc.get('specificEpithet', '')
    name = f"{genus} {species}".strip() or 'Unknown species'

    # Add individual info if available
    individual_id = enc.get('individualId')
    individual_name = enc.get('individualDisplayName', 'Unassigned')

    if individual_id:
        individual_info = f"Individual: {individual_name}<br>"
    else:
        individual_info = "Individual: Unassigned<br>"

    return Popup(
        children=[HTML(
            value=f"<b>{name}</b><br>"
            f"{individual_info}"
            f"Year: {enc.get('year', 'N/A')}<br>"
            f"Location: {enc.get('verbatimLocality', 'N/A')}<br>"
            f"<small>Encounter ID: {enc.get('id', '')}</small>"
        )],
        max_width=300
    )

In [9]:
# Filter to only encounters that have geographic coordinates
# Not all encounters have location data, so we need to check for valid lat/lon

mapped = [
    e for e in map_encounters
    if isinstance(e.get('locationGeoPoint'), dict)
    and 'lat' in e['locationGeoPoint']
    and 'lon' in e['locationGeoPoint']
]

print(f"\n{len(mapped)} of {len(map_encounters)} encounters have geographic coordinates")

# Count encounters per individual for the map summary
individual_map_counts = Counter(e.get('individualId') for e in mapped)

print("\nEncounters per individual on map:")
for ind_id in top_5_individuals:
    count = individual_map_counts[ind_id]
    details = individual_details.get(ind_id, {})
    display_name = details.get('display_name', ind_id)
    print(f"  {display_name}: {count} encounter(s)")


3 of 33 encounters have geographic coordinates

Encounters per individual on map:
  A-A-C-0007a-a-a-a-a-a-a-a-a-a-a-a-: 0 encounter(s)
  A-A-C-0005: 0 encounter(s)
  sitaaa: 0 encounter(s)
  A-A-C-0015: 2 encounter(s)
  BGBI_22-2100: 1 encounter(s)


In [10]:
# Create the interactive map with color-coded markers
# Each individual gets a unique color so you can see their movement patterns

if mapped:
    # Get coordinate bounds for fitting the map view
    lats = [e['locationGeoPoint']['lat'] for e in mapped]
    lons = [e['locationGeoPoint']['lon'] for e in mapped]

    # Create the base map
    m = Map()

    # Create a colored circle marker for each encounter
    # Same individual = same color across all their encounters
    markers = [
        CircleMarker(
            location=[e['locationGeoPoint']['lat'], e['locationGeoPoint']['lon']],
            radius=8,
            color=generate_color(e.get('individualId')),
            fill_color=generate_color(e.get('individualId')),
            fill_opacity=0.7,
            weight=2,
            popup=create_popup(e)
        )
        for e in mapped
    ]

    # Add all markers to the map
    for marker in markers:
        m.add_layer(marker)

    # Fit the map view to show all markers
    m.fit_bounds([[min(lats), min(lons)], [max(lats), max(lons)]])
else:
    # No mapped encounters - show world view
    m = Map(center=[0, 0], zoom=2)
    print("No encounters with coordinates to map")

m

Map(center=[0.0, 0.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

In [11]:
# Display a legend showing which color represents each individual
# This makes it easy to identify individuals on the map

# Create legend items as widget boxes
legend_items = []
for ind_id in top_5_individuals:
    color = generate_color(ind_id)
    details = individual_details.get(ind_id, {})
    display_name = details.get('display_name', ind_id)
    species_name = details.get('species', 'Unknown')
    encounter_count = individual_map_counts.get(ind_id, 0)

    # Color swatch
    swatch = HTML(
        value=f'<div style="width: 20px; height: 20px; background-color: {color}; '
              f'border: 2px solid #333; border-radius: 50%;"></div>'
    )

    # Label
    label = HTML(
        value=f'<b>{display_name}</b> ({species_name}) - {encounter_count} encounters'
    )

    # Row
    row = HBox([swatch, label], layout=Layout(margin='5px 0'))
    legend_items.append(row)

# Combine into legend box
title = HTML('<h3 style="margin-top: 0;">Individual Legend</h3>')
legend = VBox(
    [title] + legend_items,
    layout=Layout(
        border='2px solid #ccc',
        border_radius='5px',
        padding='10px',
        background='white'
    )
)

legend

In [10]:
# Clean up: logout when done
client.logout()

True